# S3 - Summary across runs

Collects every S1 and S2 run folder, converts the S2 rates to e- / px / s and e- / mm² / s with
the latest S1 calibration of the same camera, and writes the tables and figures for the paper to
`data/slim_summary/`:

- `leaves.csv`: one row per leaf (baseline, response, ratio, difference);
- `groups.csv`: per camera, optics and condition, n, ratio mean ± SD, geometric mean,
  difference mean ± SD;
- `calibration.csv`: gain, read noise, linearity, sample-plane reference per S1 run; the
  reference total compares the 4x/0.20 and the 50/0.95 relay;
- `figures/paired.png`, `figures/reference.png`.

Redraws runs that have no `summary.json` yet (set their ROIs first in S2).

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import slim

SIMULATE = True            # summarise the simulated runs under data/simulated/
INCLUDE_DRY = SIMULATE     # dry runs are left out of real summaries
DATA_ROOT = slim.DATA_ROOT / "simulated" if SIMULATE else slim.DATA_ROOT
OUT = DATA_ROOT / "slim_summary"

cal, leaves = slim.collect(DATA_ROOT, include_dry=INCLUDE_DRY)
stats = slim.group_stats(leaves)
print(f"{len(cal)} S1 runs, {len(leaves)} S2 runs")

In [ ]:
(OUT / "figures").mkdir(parents=True, exist_ok=True)
for name, fig in (("paired", slim.fig_paired(leaves)), ("reference", slim.fig_reference(cal))):
    if fig is not None:
        fig.savefig(OUT / "figures" / f"{name}.png", dpi=150)
slim.write_summary(OUT, cal, leaves, stats)
plt.show()
print((OUT / "README.md").read_text())

In [ ]:
for r in cal:
    print(f"{r['run']:40s} {r['camera']:8s} {r['optics']!s:14s} e-/ADU {slim.bp._fmt(r['e_per_adu']):>8s} "
          f"read {slim.bp._fmt(r['read_noise_e']):>6s} e-  reference {slim.bp._fmt(r['ref_total_e_s'])} e-/s")